Importing all necessary libraries

In [ ]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, median_absolute_error
from sklearn.model_selection import ParameterSampler

Importing features.parquet file

In [ ]:
features_df = pd.read_parquet("data/processed/features.parquet")

In [ ]:
features_df.head(5)

Sorting the DataFrame based on Time

In [ ]:
features_df = (
    features_df
    .sort_values("taxi_start")
    .reset_index(drop=True)
)

Defining X (Features)

In [ ]:
X = features_df[["typecode",
                 "queue_size",
                 "avg_taxi_out_time_1h",
                 "n_intersections_passed",
                 "sum_seg_length_m",
                 "sum_p10_hist_seg_time",
                 "sum_median_hist_seg_time",
                 "sum_p90_hist_seg_time",
                 "pushback",
                 "TO_runway_28",
                 "TO_runway_32",
                 "minute_of_day_sin",
                 "minute_of_day_cos",
                 "temp_at_taxi_start",
]]

Defining Y (Target Variable)

In [ ]:
y = features_df[["taxi_time_min"]]

We perform a time-series-cross-validation. For this, we first have to define the folds, each consisting of training, validation, and test data


In [ ]:
folds = [
    {
        "fold": 1,
        "val_start": "2025-03-01",
        "val_end": "2025-04-01",
        "test_start": "2025-04-01",
        "test_end": "2025-05-01",
    },
    {
        "fold": 2,
        "val_start": "2025-05-01",
        "val_end": "2025-06-01",
        "test_start": "2025-06-01",
        "test_end": "2025-07-01",
    },
    {
        "fold": 3,
        "val_start": "2025-07-01",
        "val_end": "2025-08-01",
        "test_start": "2025-08-01",
        "test_end": "2025-09-01",
    },
    {
        "fold": 4,
        "val_start": "2025-09-01",
        "val_end": "2025-10-01",
        "test_start": "2025-10-01",
        "test_end": "2025-11-01",
    },
    {
        "fold": 5,
        "val_start": "2025-11-01",
        "val_end": "2025-12-01",
        "test_start": "2025-12-01",
        "test_end": "2026-01-01",
    },
]

In a next step, we evaluate the model only based on training and validation data. This way, feature engineering can be performed

In [ ]:
results = []

for fold_config in folds:

    fold = fold_config["fold"]
    val_start = fold_config["val_start"]
    val_end = fold_config["val_end"]

    print(f"\n========== FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_start

    val_mask = (
        (features_df["taxi_start"] >= val_start) &
        (features_df["taxi_start"] < val_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_val = features_df.loc[val_mask, X.columns]
    y_val = features_df.loc[val_mask, "taxi_time_min"]

    print(f"Training samples:   {len(X_train):,}")
    print(f"Validation samples: {len(X_val):,}")

    model = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        n_estimators=3000,
        early_stopping_rounds=50,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    rmse = root_mean_squared_error(y_val, y_pred)
    r_2 = r2_score(y_val, y_pred)

    results.append({
        "fold": fold,
        "mae": mae,
        "rmse": rmse,
        "r2": r_2
    })

results_df = pd.DataFrame(results)

mean_row = pd.DataFrame([{
    "fold": "Average",
    "mae": results_df["mae"].mean(),
    "rmse": results_df["rmse"].mean(),
    "r2": results_df["r2"].mean()
}])

results_df = pd.concat(
    [results_df, mean_row],
    ignore_index=True
)

results_df


For each fold, hyperparameter tuning was performed separately using only the corresponding training and validation periods. By saving the best hyperparameters per fold seperately, we avoid data leakage. Otherwise, if we selected hyperparameters globally for all folds, earlier folds may have optimized parameters based on information that wasn't available at that time.

Twenty random hyperparameter combinations were sampled with `ParameterSampler` and evaluated on the validation set of the respective fold. The combination achieving the lowest validation RMSE was stored as the optimal hyperparameter configuration for that fold.

In [ ]:
param_grid = {
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.5, 1, 2],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [1, 2, 5],
    "learning_rate": [0.03, 0.05, 0.1],
}

param_samples = list(ParameterSampler(
    param_grid,
    n_iter=20,
    random_state=42
))

tuning_results = []
best_params_by_fold = {}

for fold_config in folds:

    fold = fold_config["fold"]
    val_start = fold_config["val_start"]
    val_end = fold_config["val_end"]

    print(f"\n========== TUNING FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_start

    val_mask = (
        (features_df["taxi_start"] >= val_start) &
        (features_df["taxi_start"] < val_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_val = features_df.loc[val_mask, X.columns]
    y_val = features_df.loc[val_mask, "taxi_time_min"]

    fold_results = []

    for i, params in enumerate(param_samples, start=1):

        print(f"Param set {i}/{len(param_samples)}")

        model = XGBRegressor(
            objective="reg:squarederror",
            enable_categorical=True,
            random_state=42,
            n_jobs=-1,
            n_estimators=3000,
            early_stopping_rounds=50,
            **params
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        result = {
            "fold": fold,
            "param_set": i,
            "val_start": val_start,
            "val_end": val_end,
            "rmse": rmse,
            "best_iteration": model.best_iteration,
            **params
        }

        tuning_results.append(result)
        fold_results.append(result)

    fold_results_df = pd.DataFrame(fold_results).sort_values("rmse")
    best_row = fold_results_df.iloc[0]

    best_params_by_fold[fold] = {
        param: best_row[param]
        for param in param_grid.keys()
    }

    print(f"\nBest params for Fold {fold}:")
    print(best_params_by_fold[fold])
    print(f"Best validation RMSE: {best_row['rmse']:.4f}")

tuning_results_df = pd.DataFrame(tuning_results)

tuning_results_df

Now we can add

In [ ]:
final_results = []

for fold_config in folds:

    fold = fold_config["fold"]

    val_end = fold_config["val_end"]
    test_start = fold_config["test_start"]
    test_end = fold_config["test_end"]

    print(f"\n========== FINAL TEST FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_end

    test_mask = (
        (features_df["taxi_start"] >= test_start) &
        (features_df["taxi_start"] < test_end)
    )

    X_train = features_df.loc[train_mask, X.columns]
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_test = features_df.loc[test_mask, X.columns]
    y_test = features_df.loc[test_mask, "taxi_time_min"]

    print(f"Training samples: {len(X_train):,}")
    print(f"Test samples:     {len(X_test):,}")

    model = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        n_estimators=3000,
        **best_params_by_fold[fold]
    )

    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)

    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = root_mean_squared_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    val_row = results_df.loc[results_df["fold"] == fold].iloc[0]

    final_results.append({
        "fold": fold,
        "test_start": test_start,
        "test_end": test_end,

        "val_mae": val_row["mae"],
        "test_mae": test_mae,
        "mae_diff": test_mae - val_row["mae"],

        "val_rmse": val_row["rmse"],
        "test_rmse": test_rmse,
        "rmse_diff": test_rmse - val_row["rmse"],

        "val_r2": val_row["r2"],
        "test_r2": test_r2,
        "r2_diff": test_r2 - val_row["r2"],
    })

final_results_df = pd.DataFrame(final_results)

average_row = pd.DataFrame([{
    "fold": "Average",
    "test_start": "",
    "test_end": "",

    "val_mae": final_results_df["val_mae"].mean(),
    "test_mae": final_results_df["test_mae"].mean(),
    "mae_diff": final_results_df["mae_diff"].mean(),

    "val_rmse": final_results_df["val_rmse"].mean(),
    "test_rmse": final_results_df["test_rmse"].mean(),
    "rmse_diff": final_results_df["rmse_diff"].mean(),

    "val_r2": final_results_df["val_r2"].mean(),
    "test_r2": final_results_df["test_r2"].mean(),
    "r2_diff": final_results_df["r2_diff"].mean(),
}])

final_results_df = pd.concat(
    [final_results_df, average_row],
    ignore_index=True
)

final_results_df

### Permutation Feature Importance on the Test Sets

Permutation Feature Importance (PFI) was calculated separately for each fold using the corresponding unseen test period. For every fold, the model was trained on the combined training and validation data using the fold-specific optimal hyperparameters. Feature importance was then determined by randomly permuting each feature 30 times and measuring the resulting increase in RMSE on the test set. The final importance values were obtained by averaging the fold-specific importances across all test folds.

In [ ]:
# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

selected_features = [
    "typecode",
    "queue_size",
    "avg_taxi_out_time_1h",
    "n_intersections_passed",
    "sum_seg_length_m",
    "sum_p10_hist_seg_time",
    "sum_median_hist_seg_time",
    "sum_p90_hist_seg_time",
    "pushback",
    "TO_runway_28",
    "TO_runway_32",
    "minute_of_day_sin",
    "minute_of_day_cos",
    "temp_at_taxi_start",
]

feature_groups = {
    "typecode": ["typecode"],
    "queue_size": ["queue_size"],
    "avg_taxi_out_time_1h": ["avg_taxi_out_time_1h"],
    "n_intersections_passed": ["n_intersections_passed"],
    "sum_seg_length_m": ["sum_seg_length_m"],
    "sum_p10_hist_seg_time": ["sum_p10_hist_seg_time"],
    "sum_median_hist_seg_time": ["sum_median_hist_seg_time"],
    "sum_p90_hist_seg_time": ["sum_p90_hist_seg_time"],
    "pushback": ["pushback"],
    "TO_runway_28": ["TO_runway_28"],
    "TO_runway_32": ["TO_runway_32"],
    "minute_of_day": ["minute_of_day_sin", "minute_of_day_cos"],
    "temp_at_taxi_start": ["temp_at_taxi_start"],
}

feature_rename_map = {
    "typecode": "Aircraft Type Designator",
    "queue_size": "Queue Size",
    "avg_taxi_out_time_1h": "Average Taxi-Out Time Previous Hour [min]",
    "n_intersections_passed": "Number of Intersections Passed",
    "sum_seg_length_m": "Sum of all Segment Lengths [m]",
    "sum_p10_hist_seg_time": "Sum of all Segment Times 10th Percentile [s]",
    "sum_median_hist_seg_time": "Sum of all Segment Times Median [s]",
    "sum_p90_hist_seg_time": "Sum of all Segment Times 90th Percentile [s]",
    "pushback": "Pushback Required",
    "TO_runway_28": "Take-Off Runway 28",
    "TO_runway_32": "Take-Off Runway 32",
    "minute_of_day": "Minute of Day",
    "temp_at_taxi_start": "Air Temperature [°C]",
}

feature_group_map = {
    "minute_of_day": "Temporal Feature",
    "temp_at_taxi_start": "Meteorological Feature",
}

for feature in feature_groups:
    if feature not in feature_group_map:
        feature_group_map[feature] = "Operational Feature"

group_colour_map = {
    "Operational Feature": "lightskyblue",
    "Temporal Feature": "darkorange",
    "Meteorological Feature": "darkgreen",
}

n_repeats = 30
rng = np.random.default_rng(42)

perm_results = []

for fold_config in folds:

    fold = fold_config["fold"]

    val_end = fold_config["val_end"]
    test_start = fold_config["test_start"]
    test_end = fold_config["test_end"]

    print(f"\n========== TEST PFI FOLD {fold} ==========")

    train_mask = features_df["taxi_start"] < val_end

    test_mask = (
        (features_df["taxi_start"] >= test_start) &
        (features_df["taxi_start"] < test_end)
    )

    X_train = features_df.loc[train_mask, selected_features].copy()
    y_train = features_df.loc[train_mask, "taxi_time_min"]

    X_test = features_df.loc[test_mask, selected_features].copy()
    y_test = features_df.loc[test_mask, "taxi_time_min"]

    X_train["typecode"] = X_train["typecode"].astype("category")
    X_test["typecode"] = X_test["typecode"].astype("category")

    model = XGBRegressor(
        objective="reg:squarederror",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1,
        n_estimators=3000,
        **best_params_by_fold[fold]
    )

    model.fit(
        X_train,
        y_train,
        verbose=False
    )

    baseline_pred = model.predict(X_test)
    baseline_rmse = root_mean_squared_error(y_test, baseline_pred)

    for feature_name, cols in feature_groups.items():

        importances = []

        for repeat in range(n_repeats):
            X_test_perm = X_test.copy()

            shuffled_idx = rng.permutation(len(X_test_perm))

            for col in cols:
                X_test_perm[col] = X_test_perm[col].to_numpy()[shuffled_idx]

            X_test_perm["typecode"] = X_test_perm["typecode"].astype("category")

            perm_pred = model.predict(X_test_perm)
            perm_rmse = root_mean_squared_error(y_test, perm_pred)

            importances.append(perm_rmse - baseline_rmse)

        perm_results.append({
            "fold": fold,
            "feature": feature_name,
            "importance_mean": np.mean(importances),
            "importance_std": np.std(importances),
            "baseline_rmse": baseline_rmse
        })

perm_importance_df = pd.DataFrame(perm_results)

perm_summary_df = (
    perm_importance_df
    .groupby("feature")
    .agg(
        mean_importance=("importance_mean", "mean"),
        std_importance=("importance_mean", "std"),
        mean_perm_std=("importance_std", "mean")
    )
    .sort_values("mean_importance", ascending=False)
    .reset_index()
)

plot_df = perm_summary_df.copy()
plot_df["label"] = plot_df["feature"].map(feature_rename_map).fillna(plot_df["feature"])
plot_df["group"] = plot_df["feature"].map(feature_group_map).fillna("Operational Feature")

plot_df = plot_df.sort_values("mean_importance", ascending=True)

fig = go.Figure()

category_order = plot_df["label"].tolist()

for group_name, sub in plot_df.groupby("group"):
    fig.add_trace(
        go.Bar(
            x=sub["mean_importance"],
            y=sub["label"],
            orientation="h",
            error_x=dict(
                type="data",
                array=sub["std_importance"],
                visible=True,
                thickness=3,
                width=10
            ),
            marker=dict(color=group_colour_map[group_name]),
            name=group_name,
            showlegend=True
        )
    )

fig.update_layout(
    height=900,
    width=1500,
    margin=dict(t=50, b=70, l=430, r=50),

    font=dict(
        family=font_family,
        color=font_colour
    ),

    title=None,

    xaxis=dict(
        title="Mean Increase in RMSE [min]",
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size)
    ),

    yaxis=dict(
        title="Features",
        title_font=dict(size=axis_title_size),
        tickfont=dict(size=axis_tick_size),
        categoryorder="array",
        categoryarray=category_order
    ),

    legend=dict(
        title="Feature Groups",
        font=dict(size=legend_font_size),
        title_font=dict(size=legend_title_size),
        x=0.99,
        y=0.01,
        xanchor="right",
        yanchor="bottom"
    ),
)

fig.show()

fig.write_image("Permutation_Feature_Importance_RMSE_Test.pdf")

perm_summary_df